In [ ]:
import pandas as pd
import numpy as np
import os
import re
import string
import nltk
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup, BertModel, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset, random_split
import torch
from tqdm import tqdm
import logging
import torch.nn as nn
import torch.optim as optim
from torch.optim import AdamW
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from transformers import LongformerForSequenceClassification
from transformers import LongformerTokenizer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from transformers import LongformerModel, LongformerConfig

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
class ChunkedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length  # NEW: max total tokens per doc

    def __len__(self):
        return len(self.texts)

    def chunk_text(self, text):
        # Tokenize, no truncation
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        # Truncate to doc_max_length (e.g., 4096)
        tokens = tokens[:self.doc_max_length]
        # Break into chunks of size chunk_size
        chunks = []
        for i in range(0, len(tokens), self.chunk_size):
            chunk = tokens[i:i+self.chunk_size]
            # Add [CLS] and [SEP]
            chunk = [self.tokenizer.cls_token_id] + chunk + [self.tokenizer.sep_token_id]
            # Pad if needed
            if len(chunk) < self.max_length:
                chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            # Truncate any overlong chunk (edge case)
            chunk = chunk[:self.max_length]
            chunks.append(chunk)
        return chunks

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])
        chunks = self.chunk_text(text)
        return {
            'chunks': torch.tensor(chunks, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float),
            'num_chunks': len(chunks)
        }


In [ ]:
def bert_collate_fn(batch):
    # Unpack
    all_chunks = [item['chunks'] for item in batch]
    all_labels = torch.tensor([item['label'] for item in batch], dtype=torch.float)
    all_num_chunks = [item['num_chunks'] for item in batch]
    # Stack chunks into flat [sum_chunks, max_length]
    flat_chunks = torch.cat(all_chunks, dim=0)
    return {
        'chunks': flat_chunks,  # [total_chunks, max_length]
        'labels': all_labels,   # [batch_size]
        'num_chunks': all_num_chunks
    }

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device, scheduler):
    model.train()
    total_loss = 0.0
    total = 0
    correct = 0
    pbar = tqdm(dataloader, desc="Training", ncols=120)
    for batch in pbar:
        chunks = batch['chunks'].to(device)
        labels = batch['labels'].to(device)
        num_chunks = batch['num_chunks']

        optimizer.zero_grad()
        outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
        logits = outputs.logits.view(-1)
        chunk_idx = 0
        pooled_preds = []
        for nc in num_chunks:
            chunk_logits = logits[chunk_idx:chunk_idx+nc]
            prob = torch.sigmoid(chunk_logits)
            pooled_pred = torch.max(prob)
            pooled_preds.append(pooled_pred)
            chunk_idx += nc
        pooled_preds = torch.stack(pooled_preds)
        loss = criterion(pooled_preds, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * len(labels)
        preds = (pooled_preds >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)

        pbar.set_postfix({
            'loss': total_loss / total if total > 0 else 0,
            'acc': 100.0 * correct / total if total > 0 else 0
        })

    avg_loss = total_loss / total
    avg_acc = 100.0 * correct / total
    print(f"[Train] Loss: {avg_loss:.4f} | Accuracy: {avg_acc:.2f}%")
    return avg_loss, avg_acc


In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    all_preds = []
    all_labels = []
    pbar = tqdm(dataloader, desc="Validating", ncols=120)
    with torch.no_grad():
        for batch in pbar:
            chunks = batch['chunks'].to(device)
            labels = batch['labels'].to(device)
            num_chunks = batch['num_chunks']

            outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
            logits = outputs.logits.view(-1)
            chunk_idx = 0
            pooled_preds = []
            for nc in num_chunks:
                chunk_logits = logits[chunk_idx:chunk_idx+nc]
                prob = torch.sigmoid(chunk_logits)
                pooled_pred = torch.max(prob)
                pooled_preds.append(pooled_pred)
                chunk_idx += nc
            pooled_preds = torch.stack(pooled_preds)
            loss = criterion(pooled_preds, labels)

            total_loss += loss.item() * len(labels)
            preds = (pooled_preds >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'val_loss': total_loss / total if total > 0 else 0,
                'val_acc': 100.0 * correct / total if total > 0 else 0
            })

    avg_loss = total_loss / total
    avg_acc = 100.0 * correct / total
    print(f"[Valid] Loss: {avg_loss:.4f} | Accuracy: {avg_acc:.2f}%")
    return avg_loss, avg_acc, all_preds, all_labels


In [ ]:
def get_predictions(model, data_loader, device, pooling='max'):
    """
    Generate predictions for a dataloader using chunked input and a pooling strategy.

    Args:
        model: Trained BERT model.
        data_loader: DataLoader using chunked collate function.
        device: 'cuda' or 'cpu'.
        pooling: 'max' (recommended), 'mean', or custom.

    Returns:
        np.ndarray: predicted labels (0/1)
        np.ndarray: true labels
        np.ndarray: document-level probabilities (float)
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            chunks = batch['chunks'].to(device)
            labels = batch['labels'].to(device)
            num_chunks = batch['num_chunks']
            outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
            logits = outputs.logits.view(-1)
            chunk_idx = 0
            pooled_probs = []
            for nc in num_chunks:
                chunk_logits = logits[chunk_idx:chunk_idx+nc]
                prob = torch.sigmoid(chunk_logits)
                if pooling == 'max':
                    pooled_prob = torch.max(prob)
                elif pooling == 'mean':
                    pooled_prob = torch.mean(prob)
                else:
                    raise ValueError(f"Unknown pooling: {pooling}")
                pooled_probs.append(pooled_prob)
                chunk_idx += nc
            pooled_probs = torch.stack(pooled_probs)
            preds = (pooled_probs >= 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(pooled_probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# 0.733145
mimic_train = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_train.csv')
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_val.csv')

In [ ]:
train_texts = mimic_train['text'].tolist()
train_labels = mimic_train['label'].tolist()
val_texts = mimic_test['text'].tolist()
val_labels = mimic_test['label'].tolist()

In [ ]:
train_dataset = ChunkedTextDataset(train_texts, train_labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096)
val_dataset = ChunkedTextDataset(val_texts, val_labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=bert_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=bert_collate_fn)

In [ ]:
# Define the optimizer and loss function
num_epochs = 12
# Calculate class counts
num_negative = (np.array(train_labels) == 0).sum()
num_positive = (np.array(train_labels) == 1).sum()
pos_weight = torch.tensor([num_negative / num_positive * 1.2], dtype=torch.float).to(device)
print(f"num_negative: {num_negative}, num_positive: {num_positive}, pos_weight: {pos_weight.item():.2f}")
#model.config.attention_probs_dropout_prob = 0.2  # Increasing attention dropout to 0.2
#model.config.hidden_dropout_prob = 0.2  # Increasing hidden dropout to 0.2
optimizer = optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

num_negative: 901, num_positive: 2402, pos_weight: 0.45


In [ ]:
save_directory_model = '/content/drive/My Drive/EHR_PROJ/MODELS/mimic_bert_0522'
os.makedirs(save_directory_model, exist_ok=True)

In [ ]:
best_val_accuracy = 0.0

for epoch in range(num_epochs):
    print(f'Epoch [{epoch + 1}/{num_epochs}]')
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
    print(f'Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy:.2f}%')

    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, criterion, device)
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%')

    # Save the model only if the validation accuracy has improved
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model.save_pretrained(save_directory_model)
        tokenizer.save_pretrained(save_directory_model)
        print(f"Model saved at epoch {epoch + 1} with improved validation accuracy: {val_accuracy:.2f}%")

        # Get predictions on the combined validation set
        predictions, true_labels, _ = get_predictions(model, val_loader, device, pooling='max')

        # Calculate confusion matrix
        cm = confusion_matrix(true_labels, predictions)
        print("Confusion Matrix:")
        print(cm)

        # Calculate precision, recall, F1-score
        report = classification_report(true_labels, predictions, digits=3)
        print("Classification Report:")
        print(report)

Epoch [1/12]


Training: 100%|█████████████████████████████████████████████████| 826/826 [08:51<00:00,  1.55it/s, loss=0.415, acc=27.7]


[Train] Loss: 0.4147 | Accuracy: 27.70%
Training Loss: 0.4147, Training Accuracy: 27.70%


Validating: 100%|███████████████████████████████████████| 207/207 [01:02<00:00,  3.30it/s, val_loss=0.412, val_acc=27.2]


[Valid] Loss: 0.4117 | Accuracy: 27.24%
Validation Loss: 0.4117, Validation Accuracy: 27.24%
Model saved at epoch 1 with improved validation accuracy: 27.24%


...

[Valid] Loss: 0.3636 | Accuracy: 74.82%
Validation Loss: 0.3636, Validation Accuracy: 74.82%
Epoch [12/12]


Training: 100%|█████████████████████████████████████████████████| 826/826 [08:50<00:00,  1.56it/s, loss=0.343, acc=80.2]


[Train] Loss: 0.3433 | Accuracy: 80.23%
Training Loss: 0.3433, Training Accuracy: 80.23%


Validating: 100%|███████████████████████████████████████| 207/207 [01:02<00:00,  3.29it/s, val_loss=0.366, val_acc=74.9]

[Valid] Loss: 0.3661 | Accuracy: 74.94%
Validation Loss: 0.3661, Validation Accuracy: 74.94%
